# 01_messages_model

对应 `index.md` 第 1 阶段：Message 类型、ChatModel 调用（真 API 或假模型）。

笔记本在 `codes/langchain/01_messages_model/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

真实模型：环境变量 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_MODEL`，默认 `deepseek-v4-flash`）。没 key 时用假模型也能完成本阶段大部分题。


In [14]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "01_messages_model" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langchain" / "01_messages_model"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT

import os
LOCAL_ENV = ROOT.parent.resolve() / "local.env"
def load_local_env() -> None:
    """Load langchain/local.env into os.environ if keys are not already set."""
    if not LOCAL_ENV.is_file():
        return
    for raw in LOCAL_ENV.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip("\"'")
        if key and key not in os.environ:
            os.environ[key] = value

load_local_env()

print("DEEPSEEK_API_KEY" in os.environ)

True


## 1. 三种消息

分别创建：

- `SystemMessage`，内容 `你是简洁的助手。`
- `HumanMessage`，内容 `用一句话介绍你自己。`
- `AIMessage`，内容 `我是助手。`

按这个顺序放进列表 `messages`，打印每个元素的类型名（`type(m).__name__`）和 `content`。


In [2]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# 作答

messages = [
    SystemMessage(content="你是简洁的助手"),
    HumanMessage(content="用一句话介绍你自己。"),
    AIMessage(content="我是助手。")
]

for m in messages:
    print(type(m).__name__, m.content)
#评阅
# 三种类型与打印对。SystemMessage 题干带句号「你是简洁的助手。」，你少了末尾的「。」。

#参考答案
# messages = [
#     SystemMessage(content="你是简洁的助手。"),
#     HumanMessage(content="用一句话介绍你自己。"),
#     AIMessage(content="我是助手。"),
# ]
# for m in messages:
#     print(type(m).__name__, m.content)

SystemMessage 你是简洁的助手
HumanMessage 用一句话介绍你自己。
AIMessage 我是助手。


## 2. 假模型一轮对话

已有 `model`（`FakeListChatModel`）和用户消息 `human`。

调用 `model.invoke(...)` 得到回复，打印回复的 `content`。


In [8]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import HumanMessage

model = FakeListChatModel(responses=["你好，我是假模型。"])
human = HumanMessage(content="你好")

# 作答

response = model.invoke([human])
print(response.content)

print("-" * 20)

from rich import print as rprint
rprint(response)
#评阅
# 对。invoke 传入消息列表，打印 content。多打 rich 结构无妨。

#参考答案
# print(model.invoke([human]).content)



你好，我是假模型。
--------------------


AIMessage(
    content='你好，我是假模型。',
    additional_kwargs={},
    response_metadata={},
    id='lc_run--01a0d261-20ce-7602-a524-5339e9ec4bd3-0',
    tool_calls=[],
    invalid_tool_calls=[]
)

## 3. 带系统提示的消息列表

组装消息列表：系统提示 `只回答一个词。`，用户问题 `天空通常是什么颜色？`。

用已有的 `model` 调用，打印模型返回的 `content`。


In [9]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import HumanMessage, SystemMessage

model = FakeListChatModel(responses=["蓝色"])

# 作答

response = model.invoke([
    SystemMessage(content="只回答一个词。"),
    HumanMessage(content="天空通常是什么颜色？")
])

print(response.content)
#评阅
# 对。系统 + 用户消息列表调用正确。

#参考答案
# msgs = [
#     SystemMessage(content="只回答一个词。"),
#     HumanMessage(content="天空通常是什么颜色？"),
# ]
# print(model.invoke(msgs).content)


蓝色


## 4. 连续两轮（假模型）

`model` 里预置了两个回复。先用 `HumanMessage("第一问")` 调一次，再把**上一轮的 AI 回复**和 `HumanMessage("第二问")` 一起放进历史再调一次。

按顺序打印两轮回复的 `content`。


In [17]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import HumanMessage

model = FakeListChatModel(responses=["答一", "答二"])

# 作答

messages = [HumanMessage(content="第一问")]
r1 = model.invoke(messages)

messages.append(r1)
messages.append(HumanMessage(content="第二问"))
r2 = model.invoke(messages)

print(r1.content)
print(r2.content)
#评阅
# 对。第一轮后把 AI 回复追加进历史，再带上第二问；两轮 content 正确。

#参考答案
# r1 = model.invoke([HumanMessage(content="第一问")])
# print(r1.content)
# r2 = model.invoke([HumanMessage(content="第一问"), r1, HumanMessage(content="第二问")])
# print(r2.content)


答一
答二


## 5. 真假模型二选一

写一个函数 `make_chat_model(model="deepseek:deepseek-v4-flash")`：

- 用 `langchain.chat_models.init_chat_model`，通过 `provider:model_name` 字符串选模型（例如 `deepseek:deepseek-v4-flash`）
- 若 `model` 以 `deepseek:` 开头且环境变量 `DEEPSEEK_API_KEY` 为空：返回 `FakeListChatModel(responses=["（离线假回复）"])`
- 否则：`return init_chat_model(model)`

用默认参数得到模型，对 `HumanMessage("ping")` 调用一次，打印返回的 `content`。


In [20]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import HumanMessage

# 作答

def make_chat_model(model: str = "deepseek:deepseek-v4-flash"):
    provider, _, _ = model.partition(":")
    if provider == "deepseek" and not os.environ.get("DEEPSEEK_API_KEY"):
        return FakeListChatModel(responses=["（离线假回复）"])
    return init_chat_model(model)

chat = make_chat_model("deepseek:deepseek-v4-flash")
print(chat.invoke([HumanMessage("ping")]).content)
#评阅
# 对。init_chat_model + deepseek: 前缀；无 key 走假模型。调用成功。

#参考答案
# def make_chat_model(model: str = "deepseek:deepseek-v4-flash"):
#     provider, _, _ = model.partition(":")
#     if provider == "deepseek" and not os.environ.get("DEEPSEEK_API_KEY"):
#         return FakeListChatModel(responses=["（离线假回复）"])
#     return init_chat_model(model)
# print(make_chat_model().invoke([HumanMessage(content="ping")]).content)


pong 🏓

What can I help you with?
